<a href="https://colab.research.google.com/github/aguadooscar/03MIAR-AlgoritmosOptimizacion/blob/main/Trabajo_Pr%C3%A1ctico_Algoritmos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Trabajo Práctico<br>
Nombre y Apellidos: Óscar Aguado Miravalles  <br>
Url: https://github.com/aguadooscar/03MIAR-AlgoritmosOptimizacion<br>
Google Colab: https://colab.research.google.com/drive/1iCFJb-xF3QqBy-EtF3O6Rat30mOK1r3g <br>
Problema:
>1. Sesiones de doblaje <br>

Descripción del problema:(copiar enunciado)

Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de grabación independientemente del número de tomas que se graben. No es posible grabar más de 6 tomas por día. El objetivo es planificar las sesiones por día de manera que el gasto por los servicios de los actores de doblaje sea el menor posible. Los datos son:

Número de actores: 10

Número de tomas : 30

Actores/Tomas : https://bit.ly/36D8IuK (archivo datos_proyecto.csv)

1 indica que el actor participa en la toma
0 en caso contrario







                                        

In [2]:
import pandas as pd
import numpy as np
import random
import math

# ============================
# 1) CARGA DE DATOS
# ============================
file_path = "Datos problema doblaje(30 tomas, 10 actores) - Hoja 1.csv"

df = pd.read_csv(file_path, skiprows=1)
df = df.drop(columns=['Unnamed: 11', 'Total'])
df = df.iloc[:30]  # Solo 30 tomas


# ============================
# 2) PARÁMETROS DEL PROBLEMA
# ============================


tomas = df.iloc[:, 1:].to_numpy()  # columnas 1..10
NUM_TOMAS = tomas.shape[0]
NUM_ACTORES = tomas.shape[1]

NUM_DIAS = 5
MAX_TOMAS_DIA = 6


# ============================================
# 3) FUNCIÓN DE COSTE = ACTOR-DÍAS
# ============================================
def coste(plan):
    """
    Calcula el coste total del plan de grabación.
    El coste se mide como número de actor-días.
    """
    total = 0
    for d in range(NUM_DIAS):
        actores_presentes = set()
        for t in plan[d]:
            actores_de_toma = np.where(tomas[t] == 1)[0]
            for a in actores_de_toma:
                actores_presentes.add(a)
        total += len(actores_presentes)
    return total


# ============================================
# 4) GENERACIÓN DE UN PLAN INICIAL FACTIBLE
# ============================================
def plan_inicial():
    """
    Genera una planificación inicial aleatoria
    distribuyendo las tomas entre los días.
    """

    lista = list(range(NUM_TOMAS))
    random.shuffle(lista)
    plan = {d: [] for d in range(NUM_DIAS)}
    idx = 0
    for d in range(NUM_DIAS):
        plan[d] = lista[idx:idx + MAX_TOMAS_DIA]
        idx += MAX_TOMAS_DIA
    return plan


# ============================================
# 5) OPERADOR DE VECINDARIO (swap entre días)
# ============================================
def vecino(plan):
    """
    Intercambia dos tomas en días distintos,
    manteniendo 6 tomas por día.
    """
    nuevo = {d: plan[d][:] for d in plan}
    d1, d2 = random.sample(range(NUM_DIAS), 2)
    i = random.randrange(MAX_TOMAS_DIA)
    j = random.randrange(MAX_TOMAS_DIA)
    nuevo[d1][i], nuevo[d2][j] = nuevo[d2][j], nuevo[d1][i]
    return nuevo


# ============================================
# 6) RECOCIDO SIMULADO
# ============================================
def recocido_simulado(temp_ini=20.0, temp_fin=0.05, alpha=0.97, iter_por_temp=250):

    #generar una solución inicial
    actual = plan_inicial()
    coste_actual = coste(actual)

    mejor = {d: actual[d][:] for d in actual}
    mejor_coste = coste_actual

    T = temp_ini

    while T > temp_fin:
        for _ in range(iter_por_temp):
            cand = vecino(actual)
            coste_cand = coste(cand)
            delta = coste_cand - coste_actual

            # Criterio de aceptación
            if delta <= 0 or random.random() < math.exp(-delta / T):
                actual = cand
                coste_actual = coste_cand

                if coste_actual < mejor_coste:
                    mejor = {d: actual[d][:] for d in actual}
                    mejor_coste = coste_actual

        T *= alpha

    return mejor, mejor_coste


# ============================
# 7) EJECUCIÓN
# ============================
mejor_plan, mejor_coste = recocido_simulado()

print("------ RESULTADO FINAL ------")
print("Coste mínimo encontrado:", mejor_coste)
print()

#mostrar la planificación final
for d in range(NUM_DIAS):
    tomas_d = sorted(mejor_plan[d])
    actores_d = set()
    for t in tomas_d:
        actores_d |= set(np.where(tomas[t] == 1)[0])
    print(f"Día {d+1}: tomas={tomas_d}, actores={sorted(a+1 for a in actores_d)}, actor-días={len(actores_d)}")


------ RESULTADO FINAL ------
Coste mínimo encontrado: 27

Día 1: tomas=[13, 17, 18, 20, 22, 23], actores=[np.int64(1), np.int64(3), np.int64(6), np.int64(8)], actor-días=4
Día 2: tomas=[2, 3, 4, 10, 14, 21], actores=[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(7), np.int64(8)], actor-días=7
Día 3: tomas=[0, 7, 9, 11, 25, 28], actores=[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(9)], actor-días=7
Día 4: tomas=[5, 6, 8, 15, 24, 29], actores=[np.int64(1), np.int64(2), np.int64(4), np.int64(5), np.int64(10)], actor-días=5
Día 5: tomas=[1, 12, 16, 19, 26, 27], actores=[np.int64(1), np.int64(3), np.int64(4), np.int64(5)], actor-días=4


#Modelo
- ¿Como represento el espacio de soluciones?
- ¿Cual es la función objetivo?
- ¿Como implemento las restricciones?

#Respuesta

1.  El espacio de soluciones del problema está formado por todas las posibles asignaciones de las tomas a los días de rodaje. Sea $T=\{1,...,30\}$ el conjuto de tomas y $D=\{1,...,5\}$ los días de rodaje. Represantamos el espacio de soluciones mediante un diccionario con 5 claves donde cada clave pertenece a $D$ y para cada una de ellas un vector de 6 posiciones con  un valor de $T$, que representa la respectiva toma para el correspondiente día.


2.  El objetivo del problema es minimizar el coste total asociado a la presencia de actores durante el rodaje. Cada actor debe acudir al rodaje todos los días en los que participa en alguna toma, independientemente del número de tomas que realice ese día. Por tanto, el coste depende del número de días en los que cada actor debe estar presente. Luego la funcion objetivo es: $$min \sum_{i \in I}(\sum_{d \in D} C x_{i,d})$$
donde $I$ es el conjunto de actores,  $D$ es el conjunto de días  y $x_{i,d}$ toma el valor $
1$ si el actor $i$ asiste el día $d$ y $0$ en caso contrario.
Considerando que los actores tengan el mismo coste diario $C$ (se reducce el valor de $C$ a $1$).


3. Restricciones:

*    $\sum_{t \in T}y_{t,d}\leq 6$ $\forall d \in D$ donde  $y_{t,d}$ toma el valor $1$ si se realiza la toma $t$ el día $d$ y $0$ en caso contrario. Restricción para realizar máximo 6 tomas por día
*   $\sum_{d \in D}y_{t,d}=1$ $\forall t \in T$. Aseguramos que todas las tomas se relaizan y que ninguna se repite en varios días.











#Análisis
- ¿Que complejidad tiene el problema?. Orden de complejidad y Contabilizar el espacio de soluciones

#Respuesta

El espacio bruto de soluciones es ·$5^30 ≈ 9,3×10^{20}$.
El espacio de soluciones respetando 6 tomas por día es $(30!)/(6!)^5 ≈ 4,19×10^{23}$.

En cuanto al algoritmo, el coste de evaluar un plan es $O(D·T·A)$,donde $D$ número de días, $T$ número de tomas y $A$ número de actores ya que
solo se examinan las tomas programadas en cada día. El operador de vecindario
tiene coste $O(1)$, pues únicamente intercambia dos tomas entre dos días.

El recocido simulado ejecuta K niveles de temperatura, donde
$K = \left\lceil \frac{\log(T_f/T_0)}{\log(\alpha)} \right\rceil$ Con los parámetros del experimento
($T_0=20$, $T_f=0,05$, $\alpha = 0,97$ entonces $K ≈ 151$, iter_por_temp $= 250$), el número total de evaluaciones es:

iter_totales $= K · $iter_por_temp $= 151 · 250 = 37750$.

Por tanto, la complejidad total del algoritmo es: $$O(K · \text{iter_por_temp} · D · T · A)$$

y, dado que D, T y A son constantes del problema, $$O(K · \text{iter_por_temp})$$.

#Diseño
- ¿Que técnica utilizo? ¿Por qué?

#Respuesta
La técnica utiliza para la resolución del problema es la metaheurística recocido simulado.

La justificación es debido a un espacio de soluciones muy grande, que impide una busqueda exhaustiva y la presencia de multiple óptimos locales no permite aplicar algoritmos de busqueda local simples. Además, el metodo de recocido simulado permite explorar el espacio de soluciones de forma más amplia y la facilidad de encontrar soluciones cercanas al óptimo global.